# Week 2-2 스켈레톤: 문자열 리스트를 관계 테이블로 바꾸기

이 노트북은 `w2_2`의 스켈레톤 버전입니다. 데이터 확인과 설명은 유지하고, 보너스 Transform의 핵심 라인만 `TODO`로 비워두었습니다.


## 실습 목표

- `credits.csv`, `keywords.csv`가 어떤 데이터인지 눈으로 확인한다
- 영화 ID 기준으로 서로 다른 원천 데이터를 같은 분석 단위로 맞춘다
- 문자열처럼 저장된 리스트 컬럼을 Python 리스트로 파싱한다
- `keywords`, `cast`, `crew`를 각각 관계 테이블로 펼친다
- 키워드-영화, 배우-키워드, 감독-키워드 관계를 확인한다
- 나중에 DB에 Load한다면 어떤 테이블로 저장할지 생각해본다


## 스켈레톤 사용법

`TODO` 셀은 바로 실행하면 에러가 날 수 있습니다. 완성본인 `w2_2.ipynb`와 비교하면서 핵심 Transform 라인을 채워보세요.


## 1. 준비

In [ ]:
import ast
from pathlib import Path

import pandas as pd


In [ ]:
DATA_DIR_CANDIDATES = [Path("../data"), Path("week2/data"), Path("data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. download_dataset.py를 먼저 실행하세요.")

RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = RAW_DIR / "movies_metadata.csv"
MOVIES_CLEAN_PATH = OUTPUT_DIR / "movies_clean_for_load.csv"
CREDITS_PATH = RAW_DIR / "credits.csv"
KEYWORDS_PATH = RAW_DIR / "keywords.csv"

DATA_DIR


## 2. w2_1 결과물 읽기

이번 보너스 실습은 `w2_1`에서 만든 `movies_clean_for_load.csv`를 기준 영화 테이블처럼 사용합니다.

`movies_clean`은 영화 1개당 1행인 신뢰성 있는 정제 테이블이고, 이후 만드는 `movie_keywords`, `movie_cast`, `movie_crew`는 이 테이블의 `movie_id`와 연결됩니다.


In [ ]:
if MOVIES_CLEAN_PATH.exists():
    movies_clean = pd.read_csv(MOVIES_CLEAN_PATH, parse_dates=["release_date"])
else:
    movies_raw = pd.read_csv(METADATA_PATH, low_memory=False)
    movies_clean = movies_raw[["id", "title", "release_date", "vote_average", "vote_count"]].copy()
    movies_clean = movies_clean.rename(columns={"id": "movie_id"})
    movies_clean["movie_id"] = pd.to_numeric(movies_clean["movie_id"], errors="coerce")
    movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")
    movies_clean["vote_average"] = pd.to_numeric(movies_clean["vote_average"], errors="coerce")
    movies_clean["vote_count"] = pd.to_numeric(movies_clean["vote_count"], errors="coerce")
    movies_clean = movies_clean.dropna(subset=["movie_id", "title"]).drop_duplicates("movie_id")
    movies_clean["movie_id"] = movies_clean["movie_id"].astype("int64")
    movies_clean["release_year"] = movies_clean["release_date"].dt.year

print("movies_clean shape:", movies_clean.shape)
movies_clean.head(3)


## 3. credits / keywords 원본 확인

`credits.csv`와 `keywords.csv`는 둘 다 영화별 부가 정보를 담고 있습니다.

- `credits`: 출연진(`cast`)과 제작진(`crew`)
- `keywords`: 영화에 붙은 키워드 목록

두 파일의 공통점은 한 영화 안에 여러 값이 리스트처럼 묶여 있다는 점입니다. 이 구조는 사람이 보기에는 괜찮지만, DB에서 집계하거나 조인하기에는 불편합니다.


In [ ]:
credits_raw = pd.read_csv(CREDITS_PATH)
keywords_raw = pd.read_csv(KEYWORDS_PATH)

print("credits_raw shape:", credits_raw.shape)
print("keywords_raw shape:", keywords_raw.shape)


In [ ]:
display(credits_raw.head(2))
display(keywords_raw.head(2))


## 4. 최소 표준화: 영화 ID 이름 맞추기

`credits.csv`와 `keywords.csv`의 `id`는 영화 ID입니다. `movies_clean`에서는 같은 의미를 `movie_id`라고 부르므로 이름을 맞춥니다.

이 단계의 목적은 예쁜 컬럼명을 만드는 것이 아니라, 이후 조인에서 같은 의미의 컬럼을 분명히 하는 것입니다.


In [ ]:
# TODO: credits_raw와 keywords_raw의 id 컬럼을 movie_id로 바꾸세요.
credits = credits_raw.copy()
keywords = keywords_raw.copy()

# TODO: movie_id를 숫자형으로 변환하고, 결측 movie_id를 제거하세요.
# TODO: movie_id를 int64로 변환하세요.

display(credits.head(2))
display(keywords.head(2))


## 5. 중복 확인 후 영화 단위로 맞추기

이번 실습에서는 영화 1개당 `credits` 1행, `keywords` 1행이 되도록 맞춘 뒤 조인합니다.

중복이 있다고 바로 에러로 보지는 않지만, 조인 전에 중복 여부를 확인하고 어떤 기준으로 정리할지 정해야 합니다. 여기서는 기본 실습처럼 첫 번째 행만 남깁니다.


In [ ]:
duplicate_summary = pd.DataFrame({
    "table": ["credits", "keywords"],
    "duplicate_movie_id_rows": [
        credits["movie_id"].duplicated(keep=False).sum(),
        keywords["movie_id"].duplicated(keep=False).sum(),
    ],
})

duplicate_summary


In [ ]:
# TODO: movie_id 기준 중복은 첫 번째 행만 남기세요.
credits_one = None
keywords_one = None

# TODO: credits_one과 keywords_one을 movie_id 기준으로 inner join 하세요.
credits_keywords = None

print("credits_keywords shape:", credits_keywords.shape)
credits_keywords[["movie_id", "keywords"]].head(3)


## 6. 문자열 리스트 파싱 helper

`cast`, `crew`, `keywords`는 리스트처럼 보이지만 CSV 안에서는 문자열입니다.

예를 들어 `keywords`의 한 칸에는 이런 값이 들어 있습니다.

`[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}]`

이 문자열을 실제 Python 리스트로 바꿔야 각 키워드의 `id`, `name`을 꺼낼 수 있습니다.


In [ ]:
def parse_list_column(value):
    if pd.isna(value):
        return []
    try:
        # TODO: 문자열로 저장된 리스트를 Python 리스트로 변환하세요.
        parsed = value
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []

sample_keywords = parse_list_column(keywords_one.iloc[0]["keywords"])
sample_keywords[:3]


## 7. movie_keywords 만들기

원본 `keywords`는 영화 1행 안에 여러 키워드가 묶인 구조입니다. 이것을 `영화-키워드 조합 1개당 1행`으로 바꿉니다.

변환 전:

`movie_id = 862, keywords = [jealousy, toy, boy, ...]`

변환 후:

`movie_id = 862, keyword_name = jealousy`

`movie_id = 862, keyword_name = toy`

이렇게 바꾸면 키워드별 영화 수, 키워드별 평점, 특정 키워드 영화 목록을 쉽게 구할 수 있습니다.


In [ ]:
def make_movie_keywords(keywords_df):
    expanded = keywords_df.copy()

    # TODO: keywords 컬럼을 parse_list_column으로 파싱하세요.
    # expanded["keywords"] = ...

    # TODO: 한 영화 안의 여러 키워드를 여러 행으로 펼치세요.
    # expanded = ...

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        # TODO: keywords 딕셔너리에서 id와 name을 꺼내세요.
        "keyword_id": None,
        "keyword_name": None,
    })

    # TODO: 결측치 제거와 타입 변환을 적용하세요.
    return result

movie_keywords = make_movie_keywords(keywords_one)

print("movie_keywords shape:", movie_keywords.shape)
movie_keywords.head(10)


## 8. 자주 등장하는 키워드

`movie_keywords`는 키워드 하나가 한 행이므로 `groupby`로 키워드별 영화 수를 바로 셀 수 있습니다.


In [ ]:
# TODO: keyword_name별 고유 movie_id 개수를 집계하세요.
top_keywords = None

top_keywords.head(20)


## 9. movie_cast 만들기

`cast`도 영화 1행 안에 여러 배우가 들어 있는 구조입니다. 이것을 `영화-배우 조합 1개당 1행`으로 바꿉니다.

결과 테이블의 grain은 다음입니다.

`movie_id + actor_id + cast_order`


In [ ]:
def make_movie_cast(credits_df):
    expanded = credits_df[["movie_id", "cast"]].copy()

    # TODO: cast 컬럼을 파싱하고 여러 행으로 펼치세요.
    # expanded["cast"] = ...
    # expanded = ...

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        # TODO: cast 딕셔너리에서 actor_id, actor_name, character, cast_order를 꺼내세요.
        "actor_id": None,
        "actor_name": None,
        "character": None,
        "cast_order": None,
    })

    # TODO: 결측치 제거와 타입 변환을 적용하세요.
    return result

movie_cast = make_movie_cast(credits_one)

print("movie_cast shape:", movie_cast.shape)
movie_cast.head(10)


## 10. movie_crew 만들기

`crew`는 제작진 정보입니다. 감독, 작가, 프로듀서, 음악, 편집 등 여러 역할이 들어 있습니다.

여기서는 전체 제작진 테이블인 `movie_crew`를 만들고, 그중 `job == "Director"`인 행만 따로 `directors`로 뽑습니다.


In [ ]:
def make_movie_crew(credits_df):
    expanded = credits_df[["movie_id", "crew"]].copy()

    # TODO: crew 컬럼을 파싱하고 여러 행으로 펼치세요.
    # expanded["crew"] = ...
    # expanded = ...

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        # TODO: crew 딕셔너리에서 crew_id, crew_name, department, job을 꺼내세요.
        "crew_id": None,
        "crew_name": None,
        "department": None,
        "job": None,
    })

    # TODO: 결측치 제거와 타입 변환을 적용하세요.
    return result

movie_crew = make_movie_crew(credits_one)

# TODO: job이 Director인 행만 directors로 추출하세요.
directors = None

print("movie_crew shape:", movie_crew.shape)
print("directors shape:", directors.shape)
movie_crew.head(10)


## 11. 키워드와 메타데이터 연결

이제 `movie_keywords`와 `movies_clean`을 `movie_id`로 연결합니다. 그러면 키워드만 있는 테이블에 영화 제목, 개봉연도, 평점 같은 메타데이터를 붙일 수 있습니다.


In [ ]:
# TODO: movie_keywords와 movies_clean을 movie_id 기준으로 연결하세요.
keyword_movies = None

keyword_movies[["keyword_name", "title", "release_year", "vote_average", "vote_count"]].head(20)


## 12. 키워드별 평균 평점 보기

키워드별로 어떤 영화들이 있고 평균 평점이 어떤지 확인합니다.

단, 영화 수가 너무 적은 키워드는 평균이 불안정하므로 여기서는 `movie_count >= 20`인 키워드만 봅니다.


In [ ]:
# TODO: keyword_name별 영화 수, 평균 평점, 총 투표 수를 집계하세요.
keyword_rating_summary = None

keyword_rating_summary.head(20)


## 13. 배우와 키워드 관계 보기

`movie_cast`와 `movie_keywords`를 연결하면 배우가 어떤 키워드의 영화에 자주 등장했는지 볼 수 있습니다.

예를 들어 특정 배우가 `martial arts`, `revenge`, `musical` 같은 키워드와 자주 연결되는지 확인할 수 있습니다.


In [ ]:
# TODO: movie_cast와 movie_keywords를 movie_id 기준으로 연결하세요.
actor_keyword = None

# TODO: actor_name + keyword_name별 영화 수를 집계하세요.
actor_keyword_summary = None

actor_keyword_summary.head(20)


## 14. 감독과 키워드 관계 보기

`directors`와 `movie_keywords`를 연결하면 감독이 자주 다루는 키워드도 볼 수 있습니다.

이 분석은 완벽한 영화 해석이라기보다, 관계 테이블로 바꿨을 때 어떤 질문을 SQL/집계로 던질 수 있는지 보여주는 예시입니다.


In [ ]:
# TODO: directors와 movie_keywords를 movie_id 기준으로 연결하세요.
director_keyword = None

# TODO: director_name + keyword_name별 영화 수를 집계하세요.
director_keyword_summary = None

director_keyword_summary.head(20)


## 15. Load로 이어지는 테이블 설계

보너스 실습의 결과는 보고용 표라기보다 DB에 저장 가능한 관계 테이블 후보입니다.

Load 전에 각 테이블의 grain과 key를 정리해두면, 어떤 중복을 허용하고 어떤 중복을 막아야 하는지 판단하기 쉬워집니다.


In [ ]:
load_plan = pd.DataFrame([
    {
        "table_name": "movie_keywords",
        "dataframe": "movie_keywords",
        "grain": "영화-키워드 조합 1개당 1행",
        "key": "movie_id + keyword_id",
    },
    {
        "table_name": "movie_cast",
        "dataframe": "movie_cast",
        "grain": "영화-배우 조합 1개당 1행",
        "key": "movie_id + actor_id + cast_order",
    },
    {
        "table_name": "movie_crew",
        "dataframe": "movie_crew",
        "grain": "영화-제작진 역할 1개당 1행",
        "key": "movie_id + crew_id + job",
    },
])

load_plan


## 16. 정리

이번 보너스 실습에서는 raw 문자열 컬럼을 그대로 두지 않고 관계 테이블로 바꿨습니다.

- `keywords` → `movie_keywords`
- `cast` → `movie_cast`
- `crew` → `movie_crew`

핵심은 `explode`라는 함수 이름이 아니라, **한 행 안에 뭉쳐 있던 여러 값을 분석 가능한 여러 행으로 바꾼다**는 Transform 감각입니다.
